In [1]:
import nltk
nltk.download('treebank')

[nltk_data] Downloading package treebank to
[nltk_data]     /Users/priyakeshri/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.


True

Assignment 1

In [2]:
import math
import random
from collections import defaultdict, Counter
from nltk.corpus import treebank

# ---------------- LOAD DATA ----------------
def load_data():
    sentences = treebank.tagged_sents()
    
    # lowercase words
    sentences = [[(w.lower(), t) for (w, t) in sent] for sent in sentences]
    
    return sentences


# ---------------- BUILD COUNTS ----------------
def build_counts(sentences):
    transition_counts = defaultdict(Counter)
    emission_counts = defaultdict(Counter)
    tag_counts = Counter()
    vocab = set()
    tags = set()

    for sentence in sentences:
        prev_tag = "<START>"

        for word, tag in sentence:
            vocab.add(word)
            tags.add(tag)

            tag_counts[tag] += 1
            emission_counts[tag][word] += 1
            transition_counts[prev_tag][tag] += 1

            prev_tag = tag

        transition_counts[prev_tag]["<END>"] += 1

    return transition_counts, emission_counts, tag_counts, vocab, tags


# ---------------- PROBABILITIES (LAPLACE) ----------------
def compute_probabilities(transition_counts, emission_counts, tag_counts, vocab, tags, alpha=1.0):
    transition_probs = defaultdict(dict)
    emission_probs = defaultdict(dict)

    all_tags = list(tags) + ["<END>"]
    V = len(vocab)

    # Transition probabilities
    for prev_tag in transition_counts:
        total = sum(transition_counts[prev_tag].values()) + alpha * len(all_tags)

        for tag in all_tags:
            count = transition_counts[prev_tag][tag]
            transition_probs[prev_tag][tag] = (count + alpha) / total

    # Emission probabilities
    for tag in tags:
        total = tag_counts[tag] + alpha * (V + 1)

        for word in vocab:
            count = emission_counts[tag][word]
            emission_probs[tag][word] = (count + alpha) / total

        emission_probs[tag]["<UNK>"] = alpha / total

    return transition_probs, emission_probs


# ---------------- RUN ASSIGNMENT 1 ----------------
sentences = load_data()

transition_counts, emission_counts, tag_counts, vocab, tags = build_counts(sentences)
transition_probs, emission_probs = compute_probabilities(
    transition_counts, emission_counts, tag_counts, vocab, tags
)

print("\n--- Assignment 1 Output ---")
print("Number of sentences:", len(sentences))
print("Number of tags:", len(tags))
print("Vocabulary size:", len(vocab))

print("\nSample Tags:", list(tags)[:10])

print("\nSample Transition Counts:")
for k in list(transition_counts.keys())[:3]:
    print(k, dict(list(transition_counts[k].items())[:5]))

print("\nSample Emission Counts:")
for k in list(emission_counts.keys())[:3]:
    print(k, dict(list(emission_counts[k].items())[:5]))


--- Assignment 1 Output ---
Number of sentences: 3914
Number of tags: 46
Vocabulary size: 11387

Sample Tags: ['POS', 'MD', 'RB', 'WP$', '$', '-LRB-', 'VBN', 'FW', 'WP', 'LS']

Sample Transition Counts:
<START> {'NNP': 774, 'DT': 905, 'IN': 505, 'PRP': 245, 'EX': 17}
NNP {'NNP': 3597, ',': 1441, 'CD': 190, 'VBZ': 345, 'VBG': 7}
, {'CD': 115, 'MD': 54, 'DT': 660, 'VBD': 268, 'NNS': 129}

Sample Emission Counts:
NNP {'pierre': 1, 'vinken': 2, 'nov.': 23, 'mr.': 375, 'elsevier': 1}
, {',': 4885, 'wa': 1}
CD {'61': 5, '29': 5, '55': 10, '30': 47, '1956': 2}


Assignment 2

In [3]:
# ---------------- VITERBI ----------------
def viterbi(sentence, tags, transition_probs, emission_probs):
    sentence = [w.lower() for w in sentence]
    tags = list(tags)

    V = [{}]
    backpointer = [{}]

    # Initialization
    for tag in tags:
        trans_p = transition_probs["<START>"].get(tag, 1e-10)
        emis_p = emission_probs[tag].get(
            sentence[0],
            emission_probs[tag].get("<UNK>", 1e-10)
        )

        V[0][tag] = math.log(trans_p) + math.log(emis_p)
        backpointer[0][tag] = None

    # Recursion
    for t in range(1, len(sentence)):
        V.append({})
        backpointer.append({})

        for curr_tag in tags:
            max_prob = float('-inf')
            best_prev = None

            emis_p = emission_probs[curr_tag].get(
                sentence[t],
                emission_probs[curr_tag].get("<UNK>", 1e-10)
            )

            for prev_tag in tags:
                trans_p = transition_probs[prev_tag].get(curr_tag, 1e-10)

                prob = V[t-1][prev_tag] + math.log(trans_p) + math.log(emis_p)

                if prob > max_prob:
                    max_prob = prob
                    best_prev = prev_tag

            V[t][curr_tag] = max_prob
            backpointer[t][curr_tag] = best_prev

    # Termination
    max_prob = float('-inf')
    best_last_tag = None

    for tag in tags:
        trans_p = transition_probs[tag].get("<END>", 1e-10)
        prob = V[-1][tag] + math.log(trans_p)

        if prob > max_prob:
            max_prob = prob
            best_last_tag = tag

    # Backtracking
    best_path = [best_last_tag]

    for t in range(len(sentence)-1, 0, -1):
        best_tag = backpointer[t][best_path[-1]]
        best_path.append(best_tag)

    best_path.reverse()
    return best_path


# ---------------- RUN ASSIGNMENT 2 ----------------
test_sentence = ["the", "market", "fell", "today"]

pred_tags = viterbi(test_sentence, tags, transition_probs, emission_probs)

print("\n--- Assignment 2 Output ---")
print("Sentence:", test_sentence)
print("Predicted Tags:", pred_tags)

print("\nWord → Tag:")
for w, t in zip(test_sentence, pred_tags):
    print(f"{w:10s} → {t}")


--- Assignment 2 Output ---
Sentence: ['the', 'market', 'fell', 'today']
Predicted Tags: ['DT', 'NN', 'VBD', '.']

Word → Tag:
the        → DT
market     → NN
fell       → VBD
today      → .


Assignment 3

In [4]:
# ---------------- SPLIT ----------------
def train_test_split(sentences, test_size=0.2):
    random.shuffle(sentences)
    split = int(len(sentences) * (1 - test_size))
    return sentences[:split], sentences[split:]


# ---------------- BASELINE ----------------
def build_baseline(emission_counts):
    word_tag = {}

    for tag in emission_counts:
        for word, count in emission_counts[tag].items():
            if word not in word_tag or count > word_tag[word][1]:
                word_tag[word] = (tag, count)

    return {w: t for w, (t, _) in word_tag.items()}


def baseline_predict(sentence, baseline, default_tag="NN"):
    return [baseline.get(w.lower(), default_tag) for w in sentence]


# ---------------- EVALUATION ----------------
def evaluate(test_data, tags, transition_probs, emission_probs, baseline):

    correct_hmm = 0
    correct_base = 0
    total = 0

    for sentence in test_data:
        if len(sentence) == 0:
            continue

        words = [w for w, t in sentence]
        true_tags = [t for w, t in sentence]

        pred_hmm = viterbi(words, tags, transition_probs, emission_probs)
        pred_base = baseline_predict(words, baseline)

        for h, b, t in zip(pred_hmm, pred_base, true_tags):
            if h == t:
                correct_hmm += 1
            if b == t:
                correct_base += 1
            total += 1

    return correct_hmm / total, correct_base / total


# ---------------- RUN ASSIGNMENT 3 ----------------
train, test = train_test_split(sentences)

# rebuild on train only
transition_counts, emission_counts, tag_counts, vocab, tags = build_counts(train)
transition_probs, emission_probs = compute_probabilities(
    transition_counts, emission_counts, tag_counts, vocab, tags
)

baseline = build_baseline(emission_counts)

acc_hmm, acc_base = evaluate(test, tags, transition_probs, emission_probs, baseline)

print("\n--- Assignment 3 Output ---")
print(f"HMM Accuracy: {acc_hmm:.4f}")
print(f"Baseline Accuracy: {acc_base:.4f}")


--- Assignment 3 Output ---
HMM Accuracy: 0.8535
Baseline Accuracy: 0.8776
